<a href="https://colab.research.google.com/github/dcangundogan/airplanefailureprediction/blob/main/Can_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ==== Core ====
import os, time
import numpy as np
import pandas as pd

# ==== Plots ====
import matplotlib.pyplot as plt
import seaborn as sns


# ==== ML utils ====
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, f1_score, roc_auc_score,
                             accuracy_score, precision_score, recall_score)

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Masking, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, History
from tensorflow.keras.metrics import AUC, Precision, Recall
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold,RandomizedSearchCV
from sklearn.metrics import make_scorer, roc_auc_score


np.random.seed(1)
tf.random.set_seed(1)


In [4]:
SEED=42;
SPLIT=0.2;
BATCH_SIZE=64;
EPOCHS=100;


In [5]:
print(tf.__version__)

2.19.0


In [6]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [7]:
# data path = /content/drive/MyDrive/data
#PREPARE THE DATA

train_FD1 = pd.read_csv("/content/drive/MyDrive/data/train_FD001.csv")
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,...,sensor_measurement_17,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21,sensor_measurement_22,sensor_measurement_23,sensor_measurement_24,sensor_measurement_25,sensor_measurement_26
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,392,2388,100.0,39.06,23.4190,NaN,NaN,NaN,NaN,NaN
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,392,2388,100.0,39.00,23.4236,NaN,NaN,NaN,NaN,NaN
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,390,2388,100.0,38.95,23.3442,NaN,NaN,NaN,NaN,NaN
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,392,2388,100.0,38.88,23.3739,NaN,NaN,NaN,NaN,NaN
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,393,2388,100.0,38.90,23.4044,NaN,NaN,NaN,NaN,NaN


In [8]:
#understand and then drop the non-unique and constant columns
np.unique(train_FD1['operational_setting_3'])




array([100.])

In [9]:
np.unique(train_FD1['sensor_measurement_1'])

array([518.67])

In [10]:
#Therefore opsetting3 is constant no need for our data
drop_list=["operational_setting_3",'sensor_measurement_1','sensor_measurement_5','sensor_measurement_6','sensor_measurement_10','sensor_measurement_14','sensor_measurement_16','sensor_measurement_18','sensor_measurement_19','sensor_measurement_22','sensor_measurement_23','sensor_measurement_24','sensor_measurement_25','sensor_measurement_26']
train_FD1.drop(drop_list,axis=1,inplace=True)

In [11]:
type(train_FD1)

pandas.core.frame.DataFrame

In [12]:
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8.4195,392,39.06,23.4190
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8.4318,392,39.00,23.4236
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8.4178,390,38.95,23.3442
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8.3682,392,38.88,23.3739
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8.4294,393,38.90,23.4044


In [13]:
rul=pd.DataFrame(train_FD1.groupby('unit_number')['time_in_cycles'].max()).reset_index()
rul.columns = ["unit_number",'max']
rul.head()

,unit_number,max
0,1,192
1,2,287
2,3,179
3,4,189
4,5,269


In [14]:
train_FD1 = train_FD1.merge(rul, on=['unit_number'], how='left')
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21,max
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8.4195,392,39.06,23.4190,192
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8.4318,392,39.00,23.4236,192
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8.4178,390,38.95,23.3442,192
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8.3682,392,38.88,23.3739,192
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8.4294,393,38.90,23.4044,192


In [15]:
train_FD1['RUL'] =train_FD1['max'] - train_FD1['time_in_cycles']
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21,max,RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8.4195,392,39.06,23.4190,192,191
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8.4318,392,39.00,23.4236,192,190
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8.4178,390,38.95,23.3442,192,189
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8.3682,392,38.88,23.3739,192,188
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8.4294,393,38.90,23.4044,192,187


In [16]:
train_FD1.drop(columns=['max'],axis=1,inplace=True)
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21,RUL
0,1,1,-0.0007,-0.0004,641.82,1589.70,1400.60,554.36,2388.06,9046.19,47.47,521.66,2388.02,8.4195,392,39.06,23.4190,191
1,1,2,0.0019,-0.0003,642.15,1591.82,1403.14,553.75,2388.04,9044.07,47.49,522.28,2388.07,8.4318,392,39.00,23.4236,190
2,1,3,-0.0043,0.0003,642.35,1587.99,1404.20,554.26,2388.08,9052.94,47.27,522.42,2388.03,8.4178,390,38.95,23.3442,189
3,1,4,0.0007,0.0000,642.35,1582.79,1401.87,554.45,2388.11,9049.48,47.13,522.86,2388.08,8.3682,392,38.88,23.3739,188
4,1,5,-0.0019,-0.0002,642.37,1582.85,1406.22,554.00,2388.06,9055.15,47.28,522.19,2388.04,8.4294,393,38.90,23.4044,187


In [17]:
type(train_FD1)

pandas.core.frame.DataFrame

In [18]:
test_FD1=pd.read_csv("/content/drive/MyDrive/data/test_FD001.csv")
test_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,sensor_measurement_1,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_5,...,sensor_measurement_17,sensor_measurement_18,sensor_measurement_19,sensor_measurement_20,sensor_measurement_21,sensor_measurement_22,sensor_measurement_23,sensor_measurement_24,sensor_measurement_25,sensor_measurement_26
0,1,1,0.0023,0.0003,100.0,518.67,643.02,1585.29,1398.21,14.62,...,392,2388,100.0,38.86,23.3735,NaN,NaN,NaN,NaN,NaN
1,1,2,-0.0027,-0.0003,100.0,518.67,641.71,1588.45,1395.42,14.62,...,393,2388,100.0,39.02,23.3916,NaN,NaN,NaN,NaN,NaN
2,1,3,0.0003,0.0001,100.0,518.67,642.46,1586.94,1401.34,14.62,...,393,2388,100.0,39.08,23.4166,NaN,NaN,NaN,NaN,NaN
3,1,4,0.0042,0.0000,100.0,518.67,642.44,1584.12,1406.42,14.62,...,391,2388,100.0,39.00,23.3737,NaN,NaN,NaN,NaN,NaN
4,1,5,0.0014,0.0000,100.0,518.67,642.51,1587.19,1401.92,14.62,...,390,2388,100.0,38.99,23.4130,NaN,NaN,NaN,NaN,NaN


In [19]:
test_FD1.drop(drop_list,axis=1,inplace=True)
test_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21
0,1,1,0.0023,0.0003,643.02,1585.29,1398.21,553.90,2388.04,9050.17,47.20,521.72,2388.03,8.4052,392,38.86,23.3735
1,1,2,-0.0027,-0.0003,641.71,1588.45,1395.42,554.85,2388.01,9054.42,47.50,522.16,2388.06,8.3803,393,39.02,23.3916
2,1,3,0.0003,0.0001,642.46,1586.94,1401.34,554.11,2388.05,9056.96,47.50,521.97,2388.03,8.4441,393,39.08,23.4166
3,1,4,0.0042,0.0000,642.44,1584.12,1406.42,554.07,2388.03,9045.29,47.28,521.38,2388.05,8.3917,391,39.00,23.3737
4,1,5,0.0014,0.0000,642.51,1587.19,1401.92,554.16,2388.01,9044.55,47.31,522.15,2388.03,8.4031,390,38.99,23.4130


In [20]:
fd1_y_true = pd.read_csv('/content/drive/MyDrive/data/RUL_FD001.txt', delim_whitespace=True,names=["RUL"])
fd1_y_true.head()

/tmp/ipython-input-2217873213.py:1: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  fd1_y_true = pd.read_csv('/content/drive/MyDrive/data/RUL_FD001.txt', delim_whitespace=True,names=["RUL"])


,RUL
0,112
1,98
2,69
3,82
4,91


In [21]:
fd1_y_true['unit_number']=fd1_y_true.index
fd1_y_true.set_index('unit_number',inplace=True)
fd1_y_true.head()

,RUL
unit_number,
0,112
1,98
2,69
3,82
4,91


In [22]:
train_FD1.dtypes

,0
unit_number,int64
time_in_cycles,int64
operational_setting_1,float64
operational_setting_2,float64
sensor_measurement_2,float64
sensor_measurement_3,float64
sensor_measurement_4,float64
sensor_measurement_7,float64
sensor_measurement_8,float64
sensor_measurement_9,float64


In [23]:
type(train_FD1)

pandas.core.frame.DataFrame

In [24]:
#normalizing ther data
#do not normalize the unit_number_time_in_cycles and RUl
feats = train_FD1.columns.drop(['unit_number', 'time_in_cycles', 'RUL'])

In [25]:
scaler = StandardScaler()#z score scaler
train_FD1[feats] = scaler.fit_transform(train_FD1[feats])
test_FD1[feats] = scaler.transform(test_FD1[feats])

In [26]:
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21,RUL
0,1,1,-0.315980,-1.372953,-1.721725,-0.134255,-0.925936,1.121141,-0.516338,-0.862813,-0.266467,0.334262,-1.058890,-0.603816,-0.781710,1.348493,1.194427,191
1,1,2,0.872722,-1.031720,-1.061780,0.211528,-0.643726,0.431930,-0.798093,-0.958818,-0.191583,1.174899,-0.363646,-0.275852,-0.781710,1.016528,1.236922,190
2,1,3,-1.961874,1.015677,-0.661813,-0.413166,-0.525953,1.008155,-0.234584,-0.557139,-1.015303,1.364721,-0.919841,-0.649144,-2.073094,0.739891,0.503423,189
3,1,4,0.324090,-0.008022,-0.661813,-1.261314,-0.784831,1.222827,0.188048,-0.713826,-1.539489,1.961302,-0.224597,-1.971665,-0.781710,0.352598,0.777792,188
4,1,5,-0.864611,-0.690488,-0.621816,-1.251528,-0.301518,0.714393,-0.516338,-0.457059,-0.977861,1.052871,-0.780793,-0.339845,-0.136018,0.463253,1.059552,187


In [51]:
threshold=50 #maybe more accurate number i am not sure
train_FD1['failure']=[1 if i<threshold else 0 for i in train_FD1.RUL]
fd1_y_true['failure'] = [1 if i <threshold else 0 for i in fd1_y_true.RUL]


In [28]:
fd1_y_true.head()

,RUL,failure
unit_number,,
0,112,0
1,98,0
2,69,0
3,82,0
4,91,0


In [29]:
train_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21,RUL,failure
0,1,1,-0.315980,-1.372953,-1.721725,-0.134255,-0.925936,1.121141,-0.516338,-0.862813,-0.266467,0.334262,-1.058890,-0.603816,-0.781710,1.348493,1.194427,191,0
1,1,2,0.872722,-1.031720,-1.061780,0.211528,-0.643726,0.431930,-0.798093,-0.958818,-0.191583,1.174899,-0.363646,-0.275852,-0.781710,1.016528,1.236922,190,0
2,1,3,-1.961874,1.015677,-0.661813,-0.413166,-0.525953,1.008155,-0.234584,-0.557139,-1.015303,1.364721,-0.919841,-0.649144,-2.073094,0.739891,0.503423,189,0
3,1,4,0.324090,-0.008022,-0.661813,-1.261314,-0.784831,1.222827,0.188048,-0.713826,-1.539489,1.961302,-0.224597,-1.971665,-0.781710,0.352598,0.777792,188,0
4,1,5,-0.864611,-0.690488,-0.621816,-1.251528,-0.301518,0.714393,-0.516338,-0.457059,-0.977861,1.052871,-0.780793,-0.339845,-0.136018,0.463253,1.059552,187,0


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['time_in_cycles'].plot(kind='hist', bins=20, title='time_in_cycles')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_2['operational_setting_1'].plot(kind='hist', bins=20, title='operational_setting_1')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_3['operational_setting_2'].plot(kind='hist', bins=20, title='operational_setting_2')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  xs = series['index']
  ys = series['sensor_measurement_4']
  
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_11.sort_values('index', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('sensor_measurement_4')

from matplotlib import pyplot as plt
_df_12['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_13['time_in_cycles'].plot(kind='line', figsize=(8, 4), title='time_in_cycles')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_14['operational_setting_1'].plot(kind='line', figsize=(8, 4), title='operational_setting_1')
plt.gca().spines[['top', 'right']].set_visible(False)

from matplotlib import pyplot as plt
_df_15['operational_setting_2'].plot(kind='line', figsize=(8, 4), title='operational_setting_2')
plt.gca().spines[['top', 'right']].set_visible(False)

In [30]:
test_FD1.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,sensor_measurement_2,sensor_measurement_3,sensor_measurement_4,sensor_measurement_7,sensor_measurement_8,sensor_measurement_9,sensor_measurement_11,sensor_measurement_12,sensor_measurement_13,sensor_measurement_15,sensor_measurement_17,sensor_measurement_20,sensor_measurement_21
0,1,1,1.055599,1.015677,0.678077,-0.853550,-1.191480,0.601408,-0.798093,-0.682579,-1.277396,0.415614,-0.919841,-0.985107,-0.781710,0.241943,0.774097
1,1,2,-1.230366,-1.031720,-1.941707,-0.338137,-1.501467,1.674769,-1.220725,-0.490117,-0.154141,1.012195,-0.502695,-1.649034,-0.136018,1.127183,0.941305
2,1,3,0.141213,0.333211,-0.441831,-0.584426,-0.843717,0.838677,-0.657216,-0.375093,-0.154141,0.754581,-0.919841,0.052112,-0.136018,1.459148,1.172256
3,1,4,1.924266,-0.008022,-0.481827,-1.044384,-0.279297,0.793483,-0.938970,-0.903570,-0.977861,-0.045381,-0.641744,-1.345067,-1.427402,1.016528,0.775945
4,1,5,0.644125,-0.008022,-0.341839,-0.543650,-0.779276,0.895170,-1.220725,-0.937081,-0.865536,0.998637,-0.919841,-1.041101,-2.073094,0.961200,1.138999


In [31]:
train_FD1.shape

(20631, 19)

In [32]:
test_FD1.shape

(13096, 17)

In [33]:

#sliding window technique thats called
#in LSTM it want 3-D numpy input data so convert the train and test data we need to make input like (batch size , timesteps,features)
def convert_3d_numpy_train(df,seq,cols):
  data= df[cols].values
  num_elements=data.shape[0]
  lstm_array=[]
  for start, stop in zip(range(0, num_elements-seq+1), range(seq, num_elements+1)):
    lstm_array.append(data[start:stop,:])
  return np.array(lstm_array)


In [34]:

def convert_3d_numpy_preds(df,seq,label):
  data= df[label].values
  num_elements=data.shape[0]
  return data[seq-1:num_elements+1]


In [35]:
def convert_3d_numpy_test(df,seq,cols,mask):
  df_mask = pd.DataFrame(np.zeros((seq-1,df.shape[1])),columns=df.columns)
  df_mask[:] = mask
  id_df1 = pd.concat([df_mask, df],ignore_index=True)
  data_array = id_df1[cols].values
  num_elements = data_array.shape[0]
  lstm_array=[]
  start = num_elements-seq
  stop = num_elements
  lstm_array.append(data_array[start:stop, :])
  return np.array(lstm_array)

In [36]:
seq_length=50
mask=0 #for zero padding

In [37]:
X_train=np.concatenate(list(list(convert_3d_numpy_train(train_FD1[train_FD1['unit_number']==unit], seq_length, feats)) for unit in train_FD1['unit_number'].unique()))
print(X_train.shape)

(15731, 50, 15)


In [38]:
#generate target of train
y_train = np.concatenate(list(list(convert_3d_numpy_preds(train_FD1[train_FD1['unit_number']==unit], seq_length, "failure")) for unit in train_FD1['unit_number'].unique()))
y_train.shape

(15731,)

In [39]:
X_test=np.concatenate(list(list(convert_3d_numpy_test(test_FD1[test_FD1['unit_number']==unit], seq_length, feats, mask)) for unit in test_FD1['unit_number'].unique()))
print(X_test.shape)

(100, 50, 15)


In [40]:
y_test = fd1_y_true.RUL.values
y_test.shape

(100,)

In [41]:
nb_features = X_train.shape[2]
nb_out = 1

In [42]:
nb_features, nb_out

(15, 1)

In [43]:
class_0 = pd.Series(y_train).value_counts()[0]
class_1 = pd.Series(y_train).value_counts()[1]
total = class_0 + class_1
class_weight = {0: class_1/total, 1: class_0/total}


In [44]:
class_weight

{0: np.float64(0.3178437480134766), 1: np.float64(0.6821562519865234)}

In [45]:
def print_results(y_test, y_pred):
    #f1-score
    f1 = f1_score(y_test, y_pred)
    print("F1 Score: ", f1)
    print(classification_report(y_test, y_pred))
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12,12))
    plt.subplot(221)
    sns.heatmap(conf_matrix, fmt="d", annot=True, cmap='Blues')
    b, t = plt.ylim()
    plt.ylim(b + 0.5, t - 0.5)
    plt.title('Confuion Matrix')
    plt.ylabel('True Values')
    plt.xlabel('Predicted Values')
    #roc_auc_score
    model_roc_auc = roc_auc_score(y_test, y_pred)
    print ("Area under curve : ",model_roc_auc,"\n")
    fpr,tpr,thresholds = roc_curve(y_test, y_pred)
    gmeans = np.sqrt(tpr * (1-fpr))
    ix = np.argmax(gmeans)
    threshold = np.round(thresholds[ix],3)
    plt.subplot(222)
    plt.plot(fpr, tpr, color='darkorange', lw=1, label = "Auc : %.3f" %model_roc_auc)
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.scatter(fpr[ix], tpr[ix], marker='o', color='black', label='Best Threshold:' + str(threshold))
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver operating characteristic')
    plt.legend(loc="lower right")

In [46]:
def train_LSTM_classifier(timesteps, features, batch_size=128, class_weight=None, mask_value=0.0):
    model = Sequential([
        Input(shape=(timesteps, features), name='input'),
        Masking(mask_value=mask_value, name='mask'),
        Bidirectional(LSTM(64, return_sequences=True, dropout=0.2, recurrent_dropout=0.2), name='bilstm_1'),
        LSTM(32, return_sequences=False, dropout=0.2, recurrent_dropout=0.2, name='lstm_2'),
        Dense(32, activation='relu', name='fc1'),
        Dropout(0.3),
        Dense(1, activation='sigmoid', name='out')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', AUC(name='auc'), AUC(name='pr_auc', curve='PR'), Precision(), Recall()]
    )
    cbs = [
    ModelCheckpoint('best.h5', monitor='val_loss', save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', patience=2, factor=0.5, verbose=1)
]
    return model, cbs


In [47]:
timesteps = X_train.shape[1]
features  = X_train.shape[2]

model, cbs = train_LSTM_classifier(timesteps, features, batch_size=128, class_weight=class_weight, mask_value=0.0)
model.summary()



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mask (Masking)                  │ (None, 50, 15)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bilstm_1 (Bidirectional)        │ (None, 50, 128)        │        40,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 32)             │        20,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ out (Dense)                     │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 62,657 (244.75 KB)

 Trainable params: 62,657 (244.75 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
model.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=128, class_weight=class_weight, callbacks=cbs, shuffle=True)

Epoch 1/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.8297 - auc: 0.9110 - loss: 0.1651 - pr_auc: 0.8389 - precision: 0.6910 - recall: 0.8639
Epoch 1: val_loss improved from inf to 0.19450, saving model to best.h5


99/99 ━━━━━━━━━━━━━━━━━━━━ 50s 357ms/step - accuracy: 0.8302 - auc: 0.9115 - loss: 0.1646 - pr_auc: 0.8397 - precision: 0.6918 - recall: 0.8644 - val_accuracy: 0.9174 - val_auc: 0.9804 - val_loss: 0.1945 - val_pr_auc: 0.9593 - val_precision: 0.7985 - val_recall: 0.9511 - learning_rate: 0.0010
Epoch 2/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 316ms/step - accuracy: 0.9247 - auc: 0.9796 - loss: 0.0799 - pr_auc: 0.9619 - precision: 0.8519 - recall: 0.9316
Epoch 2: val_loss improved from 0.19450 to 0.17026, saving model to best.h5


99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 333ms/step - accuracy: 0.9247 - auc: 0.9796 - loss: 0.0799 - pr_auc: 0.9619 - precision: 0.8519 - recall: 0.9316 - val_accuracy: 0.9272 - val_auc: 0.9835 - val_loss: 0.1703 - val_pr_auc: 0.9680 - val_precision: 0.8217 - val_recall: 0.9522 - learning_rate: 0.0010
Epoch 3/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.9346 - auc: 0.9829 - loss: 0.0726 - pr_auc: 0.9699 - precision: 0.8676 - recall: 0.9442
Epoch 3: val_loss improved from 0.17026 to 0.14838, saving model to best.h5


99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 337ms/step - accuracy: 0.9346 - auc: 0.9829 - loss: 0.0725 - pr_auc: 0.9699 - precision: 0.8675 - recall: 0.9442 - val_accuracy: 0.9256 - val_auc: 0.9902 - val_loss: 0.1484 - val_pr_auc: 0.9780 - val_precision: 0.8061 - val_recall: 0.9744 - learning_rate: 0.0010
Epoch 4/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.9363 - auc: 0.9864 - loss: 0.0636 - pr_auc: 0.9741 - precision: 0.8653 - recall: 0.9542
Epoch 4: val_loss did not improve from 0.14838
99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 336ms/step - accuracy: 0.9364 - auc: 0.9865 - loss: 0.0635 - pr_auc: 0.9742 - precision: 0.8654 - recall: 0.9541 - val_accuracy: 0.9288 - val_auc: 0.9902 - val_loss: 0.1591 - val_pr_auc: 0.9790 - val_precision: 0.8171 - val_recall: 0.9678 - learning_rate: 0.0010
Epoch 5/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step - accuracy: 0.9412 - auc: 0.9877 - loss: 0.0612 - pr_auc: 0.9768 - precision: 0.8764 - recall: 0.9550
Epoch 5: val_loss improved from 0.14838 to 0.13027, savin

99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 338ms/step - accuracy: 0.9412 - auc: 0.9877 - loss: 0.0612 - pr_auc: 0.9768 - precision: 0.8765 - recall: 0.9550 - val_accuracy: 0.9441 - val_auc: 0.9890 - val_loss: 0.1303 - val_pr_auc: 0.9760 - val_precision: 0.8763 - val_recall: 0.9367 - learning_rate: 0.0010
Epoch 6/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step - accuracy: 0.9513 - auc: 0.9894 - loss: 0.0555 - pr_auc: 0.9794 - precision: 0.9016 - recall: 0.9555
Epoch 6: val_loss did not improve from 0.13027
99/99 ━━━━━━━━━━━━━━━━━━━━ 34s 338ms/step - accuracy: 0.9513 - auc: 0.9894 - loss: 0.0555 - pr_auc: 0.9794 - precision: 0.9016 - recall: 0.9555 - val_accuracy: 0.9460 - val_auc: 0.9881 - val_loss: 0.1441 - val_pr_auc: 0.9680 - val_precision: 0.8786 - val_recall: 0.9411 - learning_rate: 0.0010
Epoch 7/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step - accuracy: 0.9482 - auc: 0.9900 - loss: 0.0541 - pr_auc: 0.9789 - precision: 0.8939 - recall: 0.9550
Epoch 7: val_loss improved from 0.13027 to 0.12647, savin

99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 337ms/step - accuracy: 0.9482 - auc: 0.9900 - loss: 0.0541 - pr_auc: 0.9790 - precision: 0.8939 - recall: 0.9550 - val_accuracy: 0.9492 - val_auc: 0.9891 - val_loss: 0.1265 - val_pr_auc: 0.9763 - val_precision: 0.8838 - val_recall: 0.9467 - learning_rate: 0.0010
Epoch 8/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 314ms/step - accuracy: 0.9517 - auc: 0.9915 - loss: 0.0508 - pr_auc: 0.9847 - precision: 0.8952 - recall: 0.9653
Epoch 8: val_loss improved from 0.12647 to 0.12038, saving model to best.h5


99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 332ms/step - accuracy: 0.9517 - auc: 0.9915 - loss: 0.0508 - pr_auc: 0.9847 - precision: 0.8952 - recall: 0.9652 - val_accuracy: 0.9514 - val_auc: 0.9899 - val_loss: 0.1204 - val_pr_auc: 0.9773 - val_precision: 0.8986 - val_recall: 0.9356 - learning_rate: 0.0010
Epoch 9/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.9517 - auc: 0.9919 - loss: 0.0490 - pr_auc: 0.9839 - precision: 0.8974 - recall: 0.9625
Epoch 9: val_loss did not improve from 0.12038
99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 336ms/step - accuracy: 0.9517 - auc: 0.9919 - loss: 0.0490 - pr_auc: 0.9839 - precision: 0.8974 - recall: 0.9624 - val_accuracy: 0.9574 - val_auc: 0.9895 - val_loss: 0.1256 - val_pr_auc: 0.9748 - val_precision: 0.9127 - val_recall: 0.9411 - learning_rate: 0.0010
Epoch 10/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.9535 - auc: 0.9922 - loss: 0.0475 - pr_auc: 0.9854 - precision: 0.8994 - recall: 0.9661
Epoch 10: val_loss did not improve from 0.12038

Epoch 1

99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 337ms/step - accuracy: 0.9546 - auc: 0.9931 - loss: 0.0454 - pr_auc: 0.9855 - precision: 0.8980 - recall: 0.9717 - val_accuracy: 0.9479 - val_auc: 0.9898 - val_loss: 0.1196 - val_pr_auc: 0.9766 - val_precision: 0.9080 - val_recall: 0.9100 - learning_rate: 5.0000e-04
Epoch 12/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 321ms/step - accuracy: 0.9581 - auc: 0.9941 - loss: 0.0428 - pr_auc: 0.9879 - precision: 0.9100 - recall: 0.9675
Epoch 12: val_loss improved from 0.11963 to 0.10759, saving model to best.h5


99/99 ━━━━━━━━━━━━━━━━━━━━ 34s 338ms/step - accuracy: 0.9581 - auc: 0.9941 - loss: 0.0428 - pr_auc: 0.9879 - precision: 0.9100 - recall: 0.9675 - val_accuracy: 0.9584 - val_auc: 0.9915 - val_loss: 0.1076 - val_pr_auc: 0.9812 - val_precision: 0.9258 - val_recall: 0.9289 - learning_rate: 5.0000e-04
Epoch 13/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.9573 - auc: 0.9946 - loss: 0.0405 - pr_auc: 0.9888 - precision: 0.9084 - recall: 0.9671
Epoch 13: val_loss did not improve from 0.10759
99/99 ━━━━━━━━━━━━━━━━━━━━ 33s 334ms/step - accuracy: 0.9573 - auc: 0.9946 - loss: 0.0405 - pr_auc: 0.9888 - precision: 0.9085 - recall: 0.9671 - val_accuracy: 0.9527 - val_auc: 0.9896 - val_loss: 0.1244 - val_pr_auc: 0.9759 - val_precision: 0.9131 - val_recall: 0.9222 - learning_rate: 5.0000e-04
Epoch 14/50
99/99 ━━━━━━━━━━━━━━━━━━━━ 0s 319ms/step - accuracy: 0.9610 - auc: 0.9941 - loss: 0.0410 - pr_auc: 0.9866 - precision: 0.9183 - recall: 0.9669
Epoch 14: val_loss did not improve from 0.1075

In [50]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")
y_pred


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step


array([[0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [1],
       [1],
       [1],
       [1],
       [0],
       [0],
       [1],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [1],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [1],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
       [0],
       [0],
       [1],
       [1],
       [0],
    

In [ ]:
print_results(fd1_y_true.failure, y_pred)

### For Train FD002 w/ same processes




In [ ]:
data_path = "/content/drive/MyDrive/data"

In [ ]:
print("Processing FD002...")
train_FD2 = pd.read_csv(f"{data_path}/train_FD002.csv")

In [ ]:
train_FD2.head()

In [ ]:
drop_list_FD2 = ['sensor_measurement_1', 'sensor_measurement_5', 'sensor_measurement_6',
                 'sensor_measurement_10', 'sensor_measurement_16', 'sensor_measurement_18',
                 'sensor_measurement_19','sensor_measurement_22','sensor_measurement_23','sensor_measurement_24','sensor_measurement_25','sensor_measurement_26']
train_FD2.drop(drop_list_FD2, axis=1, inplace=True)

In [ ]:
rul_FD2 = pd.DataFrame(train_FD2.groupby('unit_number')['time_in_cycles'].max()).reset_index()
rul_FD2.columns = ["unit_number", 'max']
train_FD2 = train_FD2.merge(rul_FD2, on=['unit_number'], how='left')
train_FD2['RUL'] = train_FD2['max'] - train_FD2['time_in_cycles']
train_FD2.drop(columns=['max'], axis=1, inplace=True)

In [ ]:
test_FD2 = pd.read_csv(f"{data_path}/test_FD002.csv")
test_FD2.drop(drop_list_FD2, axis=1, inplace=True)

In [ ]:
fd2_y_true = pd.read_csv(f'{data_path}/RUL_FD002.txt', delim_whitespace=True, names=["RUL"])
fd2_y_true['unit_number'] = fd2_y_true.index + 1
fd2_y_true.set_index('unit_number', inplace=True)

In [ ]:
feats_FD2 = train_FD2.columns.drop(['unit_number', 'time_in_cycles', 'RUL'])
scaler_FD2 = StandardScaler()
train_FD2[feats_FD2] = scaler_FD2.fit_transform(train_FD2[feats_FD2])
test_FD2[feats_FD2] = scaler_FD2.transform(test_FD2[feats_FD2])

In [ ]:
np.mean(rul_FD2['max'])

In [ ]:
train_FD2['failure'] = [1 if i < threshold else 0 for i in train_FD2.RUL]
fd2_y_true['failure'] = [1 if i < threshold else 0 for i in fd2_y_true.RUL]

In [ ]:
print(f"FD002 - Train shape: {train_FD2.shape}, Test shape: {test_FD2.shape}")

In [ ]:
# Prepare training data
X_train_FD2 = np.concatenate(list(list(convert_3d_numpy_train(train_FD2[train_FD2['unit_number']==unit], seq_length, feats_FD2)) for unit in train_FD2['unit_number'].unique()))
print(f"X_train_FD2 shape: {X_train_FD2.shape}")

In [ ]:
y_train_FD2 = np.concatenate(list(list(convert_3d_numpy_preds(train_FD2[train_FD2['unit_number']==unit], seq_length, "failure")) for unit in train_FD2['unit_number'].unique()))
print(f"y_train_FD2 shape: {y_train_FD2.shape}")


In [ ]:
X_test_FD2 = np.concatenate(list(list(convert_3d_numpy_test(test_FD2[test_FD2['unit_number']==unit], seq_length, feats_FD2, mask)) for unit in test_FD2['unit_number'].unique()))
print(f"X_test_FD2 shape: {X_test_FD2.shape}")

In [ ]:
y_test_FD2 = fd2_y_true.RUL.values
print(f"y_test_FD2 shape: {y_test_FD2.shape}")

In [ ]:
class_0_FD2 = pd.Series(y_train_FD2).value_counts()[0]
class_1_FD2 = pd.Series(y_train_FD2).value_counts()[1]
total_FD2 = class_0_FD2 + class_1_FD2
class_weight_FD2 = {0: class_1_FD2/total_FD2, 1: class_0_FD2/total_FD2}
print(f"Class weights FD2: {class_weight_FD2}")

In [ ]:
timesteps_FD2 = X_train_FD2.shape[1]
features_FD2 = X_train_FD2.shape[2]

In [ ]:
model_FD2, cbs_FD2 = train_LSTM_classifier(timesteps_FD2, features_FD2, batch_size=128, class_weight=class_weight_FD2, mask_value=0.0)
model_FD2.summary()

In [ ]:
history_FD2 = model_FD2.fit(X_train_FD2, y_train_FD2, validation_split=0.2, epochs=50, batch_size=128, class_weight=class_weight_FD2, callbacks=cbs_FD2, shuffle=True)


In [ ]:
# Predictions and evaluation
y_pred_FD2 = (model_FD2.predict(X_test_FD2) > 0.5).astype("int32")
print("\n=== FD002 Results ===")
print_results(fd2_y_true.failure, y_pred_FD2)

### FD003

In [ ]:
print("\nProcessing FD003...")
train_FD3 = pd.read_csv(f"{data_path}/train_FD003.csv")

In [ ]:
train_FD3.head()

In [ ]:
drop_list_FD3 = ['operational_setting_3', 'sensor_measurement_1', 'sensor_measurement_5',
                 'sensor_measurement_6', 'sensor_measurement_10', 'sensor_measurement_14',
                 'sensor_measurement_16', 'sensor_measurement_18', 'sensor_measurement_19',
                 'sensor_measurement_22', 'sensor_measurement_23', 'sensor_measurement_24',
                 'sensor_measurement_25', 'sensor_measurement_26']
train_FD3.drop(drop_list_FD3, axis=1, inplace=True)

In [ ]:
rul_FD3 = pd.DataFrame(train_FD3.groupby('unit_number')['time_in_cycles'].max()).reset_index()
rul_FD3.columns = ["unit_number", 'max']
train_FD3 = train_FD3.merge(rul_FD3, on=['unit_number'], how='left')
train_FD3['RUL'] = train_FD3['max'] - train_FD3['time_in_cycles']
train_FD3.drop(columns=['max'], axis=1, inplace=True)

In [ ]:
test_FD3 = pd.read_csv(f"{data_path}/test_FD003.csv")
test_FD3.drop(drop_list_FD3, axis=1, inplace=True)

In [ ]:
fd3_y_true = pd.read_csv(f'{data_path}/RUL_FD003.txt', delim_whitespace=True, names=["RUL"])
fd3_y_true['unit_number'] = fd3_y_true.index + 1
fd3_y_true.set_index('unit_number', inplace=True)

In [ ]:
feats_FD3 = train_FD3.columns.drop(['unit_number', 'time_in_cycles', 'RUL'])
scaler_FD3 = StandardScaler()
train_FD3[feats_FD3] = scaler_FD3.fit_transform(train_FD3[feats_FD3])
test_FD3[feats_FD3] = scaler_FD3.transform(test_FD3[feats_FD3])

In [ ]:
train_FD3['failure'] = [1 if i < threshold else 0 for i in train_FD3.RUL]
fd3_y_true['failure'] = [1 if i < threshold else 0 for i in fd3_y_true.RUL]

In [ ]:
print(f"FD003 - Train shape: {train_FD3.shape}, Test shape: {test_FD3.shape}")

In [ ]:
X_train_FD3 = np.concatenate(list(list(convert_3d_numpy_train(train_FD3[train_FD3['unit_number']==unit], seq_length, feats_FD3)) for unit in train_FD3['unit_number'].unique()))
print(f"X_train_FD3 shape: {X_train_FD3.shape}")

In [ ]:
X_train_FD3 = np.concatenate(list(list(convert_3d_numpy_train(train_FD3[train_FD3['unit_number']==unit], seq_length, feats_FD3)) for unit in train_FD3['unit_number'].unique()))
print(f"X_train_FD3 shape: {X_train_FD3.shape}")

In [ ]:
y_train_FD3 = np.concatenate(list(list(convert_3d_numpy_preds(train_FD3[train_FD3['unit_number']==unit], seq_length, "failure")) for unit in train_FD3['unit_number'].unique()))
print(f"y_train_FD3 shape: {y_train_FD3.shape}")

In [ ]:
X_test_FD3 = np.concatenate(list(list(convert_3d_numpy_test(test_FD3[test_FD3['unit_number']==unit], seq_length, feats_FD3, mask)) for unit in test_FD3['unit_number'].unique()))
print(f"X_test_FD3 shape: {X_test_FD3.shape}")

In [ ]:
y_test_FD3 = fd3_y_true.RUL.values
print(f"y_test_FD3 shape: {y_test_FD3.shape}")

In [ ]:
class_0_FD3 = pd.Series(y_train_FD3).value_counts()[0]
class_1_FD3 = pd.Series(y_train_FD3).value_counts()[1]
total_FD3 = class_0_FD3 + class_1_FD3
class_weight_FD3 = {0: class_1_FD3/total_FD3, 1: class_0_FD3/total_FD3}
print(f"Class weights FD3: {class_weight_FD3}")

In [ ]:
# Build and train model
timesteps_FD3 = X_train_FD3.shape[1]
features_FD3 = X_train_FD3.shape[2]

In [ ]:
model_FD3, cbs_FD3 = train_LSTM_classifier(timesteps_FD3, features_FD3, batch_size=128, class_weight=class_weight_FD3, mask_value=0.0)
model_FD3.summary()

In [ ]:
history_FD3 = model_FD3.fit(X_train_FD3, y_train_FD3, validation_split=0.2, epochs=50, batch_size=128, class_weight=class_weight_FD3, callbacks=cbs_FD3, shuffle=True)


In [ ]:
# Predictions and evaluation
y_pred_FD3 = (model_FD3.predict(X_test_FD3) > 0.5).astype("int32")
print("\n=== FD003 Results ===")
print_results(fd3_y_true.failure, y_pred_FD3)

### FD004

In [ ]:
print("\nProcessing FD004...")
train_FD4 = pd.read_csv(f"{data_path}/train_FD004.csv")

In [ ]:
train_FD4.head()

In [ ]:
drop_list_FD4 = ['sensor_measurement_1', 'sensor_measurement_5', 'sensor_measurement_6',
                 'sensor_measurement_10', 'sensor_measurement_16', 'sensor_measurement_18',
                 'sensor_measurement_19','sensor_measurement_22','sensor_measurement_23','sensor_measurement_24','sensor_measurement_25','sensor_measurement_26']
train_FD4.drop(drop_list_FD4, axis=1, inplace=True)

In [ ]:
rul_FD4 = pd.DataFrame(train_FD4.groupby('unit_number')['time_in_cycles'].max()).reset_index()
rul_FD4.columns = ["unit_number", 'max']
train_FD4 = train_FD4.merge(rul_FD4, on=['unit_number'], how='left')
train_FD4['RUL'] = train_FD4['max'] - train_FD4['time_in_cycles']
train_FD4.drop(columns=['max'], axis=1, inplace=True)

In [ ]:
test_FD4 = pd.read_csv(f"{data_path}/test_FD004.csv")
test_FD4.drop(drop_list_FD4, axis=1, inplace=True)


In [ ]:
fd4_y_true = pd.read_csv(f'{data_path}/RUL_FD004.txt', delim_whitespace=True, names=["RUL"])
fd4_y_true['unit_number'] = fd4_y_true.index + 1
fd4_y_true.set_index('unit_number', inplace=True)

In [ ]:
feats_FD4 = train_FD4.columns.drop(['unit_number', 'time_in_cycles', 'RUL'])
scaler_FD4 = StandardScaler()
train_FD4[feats_FD4] = scaler_FD4.fit_transform(train_FD4[feats_FD4])
test_FD4[feats_FD4] = scaler_FD4.transform(test_FD4[feats_FD4])


In [ ]:
train_FD4['failure'] = [1 if i < threshold else 0 for i in train_FD4.RUL]
fd4_y_true['failure'] = [1 if i < threshold else 0 for i in fd4_y_true.RUL]

In [ ]:
print(f"FD004 - Train shape: {train_FD4.shape}, Test shape: {test_FD4.shape}")

In [ ]:
X_train_FD4 = np.concatenate(list(list(convert_3d_numpy_train(train_FD4[train_FD4['unit_number']==unit], seq_length, feats_FD4)) for unit in train_FD4['unit_number'].unique()))
print(f"X_train_FD4 shape: {X_train_FD4.shape}")


In [ ]:
y_train_FD4 = np.concatenate(list(list(convert_3d_numpy_preds(train_FD4[train_FD4['unit_number']==unit], seq_length, "failure")) for unit in train_FD4['unit_number'].unique()))
print(f"y_train_FD4 shape: {y_train_FD4.shape}")

In [ ]:
X_test_FD4 = np.concatenate(list(list(convert_3d_numpy_test(test_FD4[test_FD4['unit_number']==unit], seq_length, feats_FD4, mask)) for unit in test_FD4['unit_number'].unique()))
print(f"X_test_FD4 shape: {X_test_FD4.shape}")

In [ ]:
y_test_FD4 = fd4_y_true.RUL.values
print(f"y_test_FD4 shape: {y_test_FD4.shape}")

In [ ]:
class_0_FD4 = pd.Series(y_train_FD4).value_counts()[0]
class_1_FD4 = pd.Series(y_train_FD4).value_counts()[1]
total_FD4 = class_0_FD4 + class_1_FD4
class_weight_FD4 = {0: class_1_FD4/total_FD4, 1: class_0_FD4/total_FD4}
print(f"Class weights FD4: {class_weight_FD4}")

In [ ]:
timesteps_FD4 = X_train_FD4.shape[1]
features_FD4 = X_train_FD4.shape[2]

In [ ]:
model_FD4, cbs_FD4 = train_LSTM_classifier(timesteps_FD4, features_FD4, batch_size=128, class_weight=class_weight_FD4, mask_value=0.0)
model_FD4.summary()

In [ ]:
history_FD4 = model_FD4.fit(X_train_FD4, y_train_FD4, validation_split=0.2, epochs=50, batch_size=128, class_weight=class_weight_FD4, callbacks=cbs_FD4, shuffle=True)


In [ ]:
y_pred_FD4 = (model_FD4.predict(X_test_FD4) > 0.5).astype("int32")
print("\n=== FD004 Results ===")
print_results(fd4_y_true.failure, y_pred_FD4)

### Average of all results

In [ ]:
def calculate_average_metrics(results_dict):
    """
    Calculate average metrics across all datasets.

    Parameters:
    -----------
    results_dict : dict
        Dictionary where keys are dataset names (e.g., 'FD001', 'FD002')
        and values are tuples of (y_true, y_pred)

    Returns:
    --------
    DataFrame with individual and average metrics
    """
    metrics_list = []

    for dataset_name, (y_true, y_pred) in results_dict.items():
        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)

        metrics_list.append({
            'Dataset': dataset_name,
            'Accuracy': acc,
            'Precision': precision,
            'Recall': recall,
            'F1-Score': f1
        })

    # Create DataFrame
    df_metrics = pd.DataFrame(metrics_list)

    # Calculate averages
    avg_metrics = {
        'Dataset': 'Average',
        'Accuracy': df_metrics['Accuracy'].mean(),
        'Precision': df_metrics['Precision'].mean(),
        'Recall': df_metrics['Recall'].mean(),
        'F1-Score': df_metrics['F1-Score'].mean()
    }

    # Append average row
    df_metrics = pd.concat([df_metrics, pd.DataFrame([avg_metrics])], ignore_index=True)

    return df_metrics


In [ ]:
results = {
    'FD001': (fd1_y_true.failure, y_pred.flatten()),
    'FD002': (fd2_y_true.failure, y_pred_FD2.flatten()),
    'FD003': (fd3_y_true.failure, y_pred_FD3.flatten()),
    'FD004': (fd4_y_true.failure, y_pred_FD4.flatten())
}

# Calculate and display metrics
metrics_summary = calculate_average_metrics(results)
print(metrics_summary)

# Display with formatting
print("\n" + "="*70)
print("SUMMARY OF ALL DATASETS")
print("="*70)
print(metrics_summary.to_string(index=False))
print("="*70)